In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from IPython.display import display
from ISLP import load_data, confusion_table
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

In [3]:
Default = load_data("Default")
predictors = ["income", "balance"]
X = Default[predictors]
y = Default["default"]

# Train the logistic model
model = LogisticRegression(C = np.inf, max_iter = 1000)
model.fit(X, y)
model.intercept_, model.coef_
coefs = pd.Series(
    np.r_[model.intercept_, model.coef_.ravel()],
    index=["intercept"] + predictors,
    name="coefficient",
)
coefs.to_frame().style.format("{:.4g}")

,coefficient
intercept,-11.54
income,2.081e-05
balance,0.005647


In [4]:
# Split the data into train and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.5, random_state = 0)
model = LogisticRegression(C = np.inf, max_iter = 1000).fit(X_train, y_train)
p_yes = model.predict_proba(X_valid)[:,1]
pred = np.where(p_yes > 0.5, "Yes", "No")
val_error = np.mean(pred != y_valid)
print(f"Validation set error: {val_error:.4f} ({val_error:.2%})")

Validation set error: 0.0290 (2.90%)


In [5]:
# Repeat this three times
rows = []
for state in [0, 1, 2]:
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.5, random_state=state)
    model = LogisticRegression(C=np.inf, max_iter=1000).fit(X_train, y_train)
    pred = model.predict(X_valid)            # same as p_yes > 0.5
    rows.append({"random_state": state,
                 "val_error": np.mean(pred != y_valid),
                 "share_default_in_valid": np.mean(y_valid == "Yes")})

errors = pd.DataFrame(rows).set_index("random_state")
errors.style.format("{:.4f}")

,val_error,share_default_in_valid
random_state,,
0,0.0290,0.0372
1,0.0250,0.0318
2,0.0248,0.0308


**Interpretation (c).** The validation set error depends on how the data is split. With three random splits it is 2.90 %, 2.50 % and 2.48 % (mean ≈ 2.63 %). The gap between the highest and lowest estimate is 21 misclassified observations out of 5000. The split with the highest error also has the most defaults in its validation set (3.72 % versus about 3.1 %), which makes sense because almost all errors are defaults that the model misses. The estimate of the test error is therefore variable, which is a known weakness of the validation set approach.

The numbers also need context. Only about 3.3 % of all customers default, so a classifier that always predicts "No" would already have an error of roughly 3.1–3.7 % on these validation sets. The logistic model is only slightly better. Finally, each model is fitted on just half of the observations, so the validation error tends to overestimate the test error of a model fitted on the full data set. Cross-validation addresses both issues.

In [7]:
# Logistic regression with dummy variable for the predictor student
X = Default[["income", "balance"]].assign(student=(Default["student"] == "Yes").astype(int))
y = (Default["default"] == "Yes").astype(int)

specs = {"without student": ["income", "balance"],
         "with student":    ["income", "balance", "student"]}

rows = []
for state in [0, 1, 2]:
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.5, random_state=state)
    row = {"random_state": state}
    for name, cols in specs.items():
        model = LogisticRegression(C=np.inf, max_iter=1000).fit(X_train[cols], y_train)
        row[name] = np.mean(model.predict(X_valid[cols]) != y_valid)
    row["share_default_in_valid"] = np.mean(y_valid)
    rows.append(row)

errors = pd.DataFrame(rows).set_index("random_state")
errors.loc["mean"] = errors.mean()
errors.style.format("{:.4f}")

,without student,with student,share_default_in_valid
random_state,,,
0,0.0290,0.0292,0.0372
1,0.0250,0.0262,0.0318
2,0.0248,0.0254,0.0308
mean,0.0263,0.0269,0.0333


**Interpretation (d).** Adding the dummy variable for `student` does not reduce the validation set error. On the same three splits as in (c) the error is 2.92 %, 2.62 % and 2.54 % (mean ≈ 2.69 %), compared with 2.90 %, 2.50 % and 2.48 % (mean ≈ 2.63 %) without it. That is 1–6 more misclassified observations out of 5000, which is smaller than the variation between splits in (c). The difference is therefore best seen as noise and not as a real deterioration.



In [12]:
from ISLP.models import ModelSpec as MS, summarize
state = 0
X = MS(["income", "balance"]).fit_transform(Default)  # adds an "intercept" column
y = Default["default"] == "Yes"                       # True/False works as 0/1

glm = sm.GLM(y, X, family=sm.families.Binomial()).fit()
summarize(glm)

,coef,std err,z,P>|z|
intercept,-11.540500,0.435000,-26.544,0.0
income,0.000021,0.000005,4.174,0.0
balance,0.005600,0.000000,24.835,0.0


In [13]:
def boot_fn(D, idx):
    D_ = D.loc[idx]                                   # the bootstrap sample of D is denoted D_
    X_ = MS(["income", "balance"]).fit_transform(D_)
    y_ = D_["default"] == "Yes"
    fit = sm.GLM(y_, X_, family=sm.families.Binomial()).fit()
    return fit.params[["income", "balance"]]

In [17]:
# Use bootstrapping to estimate standard errors of the logistic regression coefficients for income and balance
def boot_SE(func, D, n=None, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0 # initialize the first and second sample moments
    n = n or D.shape[0]                                  # n = 10 000 by default
    for _ in range(B):
        idx = rng.choice(D.index, n, replace=True)       # sampling with replacement
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)        # standard deviation over the B fits

boot_se = boot_SE(boot_fn, Default, B=1000, seed=0)
boot_se.to_frame("bootstrap SE").style.format("{:.4g}")


,bootstrap SE
income,4.767e-06
balance,0.0002304


In [18]:
comparison = pd.DataFrame({
    "sm.GLM SE": glm.bse[["income", "balance"]],
    "bootstrap SE": boot_se,
})
comparison["ratio"] = comparison["bootstrap SE"] / comparison["sm.GLM SE"]
comparison.style.format("{:.4g}")

,sm.GLM SE,bootstrap SE,ratio
income,4.985e-06,4.767e-06,0.9563
balance,0.0002274,0.0002304,1.013


**Interpretation (d).** The two methods give very similar standard errors. For `balance` the bootstrap estimate is about 1 % larger than the `sm.GLM()` estimate (2.30e-04 vs 2.27e-04), and for `income` it is about 4 % smaller (4.77e-06 vs 4.99e-06). Both differences are small compared with the coefficients themselves, so the conclusions about significance do not change.

The `sm.GLM()` standard errors come from maximum likelihood theory. They assume that the logistic model is correctly specified and rely on a large-sample approximation. The bootstrap makes no such assumptions; it estimates the variability directly by refitting the model on 1000 resamples of the data. That the two agree so closely suggests that the logistic model with `income` and `balance` is a reasonable description of the data, and that the formula-based standard errors can be trusted here. The small remaining differences are partly due to Monte Carlo error from using a finite number of bootstrap samples (B = 1000); a different seed would change the bootstrap estimates slightly.

In [21]:
1 - (1 - 1/10_000) ** 10_000

0.6321389535670295